In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path(r"D:\Projects\AgriRisk and ROI Prediction\Dump")

raw_file = project_root / "data" / "raw" / "upag" / "upag_complete_2023_25.csv"

upag_harmonized_file = (
    project_root / "data" / "processed" / "upag_harmonized.csv"
)

unified_file = (
    project_root / "data" / "processed" / "unified"
    / "unified_crop_yield_2013_2025.csv"
)

output_dir = project_root / "data" / "processed" / "upag"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw file:", raw_file)
print("Existing UPAg file:", upag_harmonized_file)
print("Existing unified file:", unified_file)

Project root: D:\Projects\AgriRisk and ROI Prediction\Dump
Raw file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\raw\upag\upag_complete_2023_25.csv
Existing UPAg file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\upag_harmonized.csv
Existing unified file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv


In [ ]:
#Loading complete upag dataset
df = pd.read_csv(raw_file)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
#Inspecting crops and seasons
print("Number of unique crops:", df["Crop"].nunique())

print("\nCrops:")
print(sorted(df["Crop"].dropna().unique()))

print("\nSeasons:")
print(df["Season"].value_counts())

In [ ]:
#Selecting the 8 missing crops
target_crops = [
    "Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Ragi",
    "Soybean",
    "Sugarcane"
]

recent_8 = df[
    df["Crop"].isin(target_crops) &
    df["Season"].eq("Total")
].copy()

print("Selected rows:", len(recent_8))

print("\nCrop counts:")
print(recent_8["Crop"].value_counts().sort_index())

print("\nYears available:")
print("2023-24 and 2024-25")

In [ ]:
#Checking that each crop has both the years
print(
    recent_8[
        ["Crop", "Area-2023-24", "Area-2024-25",
         "Production-2023-24", "Production-2024-25",
         "Yield-2023-24", "Yield-2024-25"]
    ].isna().sum()
)

print("\nRows per crop:")
print(
    recent_8.groupby("Crop").size().sort_index()
)

In [ ]:
#Converting wide data to long format
records = []

for _, row in recent_8.iterrows():

    for year in ["2023-24", "2024-25"]:

        records.append({
            "year": year,
            "state": row["State"],
            "district": row["District"],
            "crop": row["Crop"],
            "season": "Annual",
            "area_ha": row[f"Area-{year}"],
            "production_tonnes": row[f"Production-{year}"],
            "yield_kg_ha": row[f"Yield-{year}"],
            "source": "UPAg"
        })

recent_long = pd.DataFrame(records)

print("Shape:", recent_long.shape)

print("\nColumns:")
print(recent_long.columns.tolist())

print("\nYears:")
print(recent_long["year"].value_counts().sort_index())

In [ ]:
#Standardizing crop names
crop_mapping = {
    "Tur": "Arhar/Tur",
    "Bajra": "Bajra",
    "Gram": "Gram",
    "Groundnut": "Groundnut",
    "Jowar": "Jowar",
    "Ragi": "Ragi",
    "Soybean": "Soyabean",
    "Sugarcane": "Sugarcane"
}

recent_long["crop"] = recent_long["crop"].replace(crop_mapping)

print("Standardized crops:")
print(sorted(recent_long["crop"].unique()))

In [ ]:
#Validating units using yeild
check = recent_long[
    recent_long["area_ha"].notna() &
    recent_long["production_tonnes"].notna() &
    recent_long["yield_kg_ha"].notna() &
    (recent_long["area_ha"] > 0)
].copy()

check["calculated_yield"] = (
    check["production_tonnes"] * 1000 / check["area_ha"]
)

check["yield_difference"] = (
    check["calculated_yield"] - check["yield_kg_ha"]
).abs()

print("Valid yield checks:", len(check))

print("\nMean absolute yield difference:",
      check["yield_difference"].mean())

print("Maximum absolute yield difference:",
      check["yield_difference"].max())

print("\nSample validation:")
display(
    check[
        ["crop", "area_ha", "production_tonnes",
         "yield_kg_ha", "calculated_yield",
         "yield_difference"]
    ].head(10)
)


In [ ]:
#Checking geography mapping
geo = pd.read_csv(upag_harmonized_file)

geo_map = (
    geo[
        ["state", "district", "lgd_statecode", "lgd_distcode"]
    ]
    .drop_duplicates()
)

print("Geography mapping rows:", len(geo_map))

print("\nDuplicate state-district mappings:")
print(
    geo_map.groupby(["state", "district"]).size()
    .loc[lambda x: x > 1]
)

In [ ]:
#Normalizing geougraphy names
def normalize_geo(x):
    return (
        str(x)
        .upper()
        .strip()
        .replace("&", "AND")
        .replace(".", "")
        .replace(",", "")
        .replace("-", " ")
    )

recent_long["state_key"] = recent_long["state"].apply(normalize_geo)
recent_long["district_key"] = recent_long["district"].apply(normalize_geo)

geo_map["state_key"] = geo_map["state"].apply(normalize_geo)
geo_map["district_key"] = geo_map["district"].apply(normalize_geo)

recent_long = recent_long.merge(
    geo_map[
        ["state_key", "district_key",
         "lgd_statecode", "lgd_distcode"]
    ],
    on=["state_key", "district_key"],
    how="left"
)

print("Total records:", len(recent_long))

print("Missing state codes:",
      recent_long["lgd_statecode"].isna().sum())

print("Missing district codes:",
      recent_long["lgd_distcode"].isna().sum())

In [ ]:
#Showing unmatched districts
unmatched = recent_long[
    recent_long["lgd_distcode"].isna()
][
    ["state", "district"]
].drop_duplicates()

print("Unmatched geographic combinations:", len(unmatched))

if len(unmatched) > 0:
    display(unmatched.head(50))
else:
    print("All districts successfully matched.")

In [ ]:
#Renaming codes and finalizing schema
recent_8_final = recent_long.rename(
    columns={
        "lgd_statecode": "state_code",
        "lgd_distcode": "district_code"
    }
).copy()

recent_8_final = recent_8_final[
    [
        "year",
        "state",
        "district",
        "state_code",
        "district_code",
        "crop",
        "season",
        "area_ha",
        "production_tonnes",
        "yield_kg_ha",
        "source"
    ]
]

print("Final schema:")
print(recent_8_final.columns.tolist())

print("\nShape:", recent_8_final.shape)

In [ ]:
#Validating duplicates
key_columns = [
    "year",
    "state",
    "district",
    "crop"
]

duplicates = recent_8_final.duplicated(
    subset=key_columns
).sum()

print("Duplicate district-crop-year records:", duplicates)

if duplicates == 0:
    print("PASS: No duplicate records.")
else:
    print("WARNING: Duplicate records found.")
    display(
        recent_8_final[
            recent_8_final.duplicated(
                subset=key_columns,
                keep=False
            )
        ].sort_values(key_columns)
    )

In [ ]:
#Coveraging validation for the 8 crops
coverage = (
    recent_8_final
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print(coverage)

print("\nExpected crops:", len(target_crops))
print("Actual standardized crops:",
      recent_8_final["crop"].nunique())

print("\nExpected years: 2")
print("Actual years:",
      recent_8_final["year"].nunique())

In [ ]:
#Missing values check
print("Missing values:")
print(recent_8_final.isna().sum())

print("\nMissing percentage:")
print(
    (recent_8_final.isna().mean() * 100)
    .round(2)
)

In [ ]:
#Saving the 8 crop upag extension
recent_8_final = recent_8_final.sort_values(
    ["year", "state", "district", "crop"]
).reset_index(drop=True)

recent_8_file = (
    output_dir / "upag_recent_8_crops_2023_2025.csv"
)

recent_8_final.to_csv(
    recent_8_file,
    index=False
)

print("Saved successfully.")
print("File:", recent_8_file)
print("Shape:", recent_8_final.shape)

In [ ]:
#Loading existing unified data
unified = pd.read_csv(unified_file)

print("Existing unified dataset:")
print("Shape:", unified.shape)

print("\nSource counts:")
print(unified["source"].value_counts())

print("\nYears:")
print(sorted(unified["year"].unique()))

print("\nCrops:")
print(sorted(unified["crop"].unique()))

In [ ]:
#Making Sure We Are Only Adding the Missing Recent Crops
existing_recent = unified[
    unified["year"].isin(["2023-24", "2024-25"])
]

print("Existing recent records:",
      len(existing_recent))

print("\nExisting recent crops:")
print(sorted(existing_recent["crop"].unique()))

print("\nNew 8-crop records:",
      len(recent_8_final))

print("\nNew crops:")
print(sorted(recent_8_final["crop"].unique()))

In [ ]:
#Merging the 8 crops
unified_updated = pd.concat(
    [unified, recent_8_final],
    ignore_index=True
)

print("Old shape:", unified.shape)
print("New shape:", unified_updated.shape)

print("\nAdded records:",
      len(unified_updated) - len(unified))

In [ ]:
#Final duplicate check
final_keys = [
    "year",
    "state",
    "district",
    "crop"
]

final_duplicates = unified_updated.duplicated(
    subset=final_keys
).sum()

print("Final duplicate records:", final_duplicates)

if final_duplicates == 0:
    print("PASS: Final dataset has no duplicate district-crop-year records.")
else:
    print("WARNING: Duplicate records found.")
    display(
        unified_updated[
            unified_updated.duplicated(
                subset=final_keys,
                keep=False
            )
        ].sort_values(final_keys).head(50)
    )
    

In [ ]:
#Final crop year coverage
final_coverage = (
    unified_updated
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print(final_coverage)

In [ ]:
#Strict 12 crops x 12 years check
expected_crops = [
    "Arhar/Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Maize",
    "Ragi",
    "Rice",
    "Soyabean",
    "Sugarcane",
    "Urad",
    "Wheat"
]

expected_years = [
    "2013-2014",
    "2014-2015",
    "2015-2016",
    "2016-2017",
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

print("Expected crops:", len(expected_crops))
print("Actual crops:", unified_updated["crop"].nunique())

print("\nExpected years:", len(expected_years))
print("Actual years:", unified_updated["year"].nunique())

missing_combinations = []

for crop in expected_crops:
    for year in expected_years:
        count = len(
            unified_updated[
                (unified_updated["crop"] == crop) &
                (unified_updated["year"] == year)
            ]
        )

        if count == 0:
            missing_combinations.append(
                (crop, year)
            )

print("\nMissing crop-year combinations:",
      len(missing_combinations))

if missing_combinations:
    print(missing_combinations)
else:
    print("PASS: Every crop has data for every year.")

In [ ]:
#Source and year summary
print("Records by source:")
print(
    unified_updated["source"]
    .value_counts()
)

print("\nRecords by year and source:")
print(
    unified_updated
    .groupby(["year", "source"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
#Final missing values
print("Final missing values:")
print(
    unified_updated[
        [
            "area_ha",
            "production_tonnes",
            "yield_kg_ha"
        ]
    ].isna().sum()
)

In [ ]:
#Correcting year format + geography mapping

# Correcting remaining UPAg → LGD district names
district_mapping = {
    "YSR Kadapa": "Y S R Kadapa",
    "Kaimur": "Kaimur Bhabua",
    "Khandwa": "Khandwa East Nimar",
    "Khargone": "Khargone West Nimar",
    "Sahibzada Ajit Singh Nagar": "S A S Nagar"
}

# Recreating geography mapping
geo = pd.read_csv(upag_harmonized_file)

geo_map = (
    geo[
        ["state", "district", "lgd_statecode", "lgd_distcode"]
    ]
    .drop_duplicates()
)

geo_map["state_key"] = geo_map["state"].apply(normalize_geo)
geo_map["district_key"] = geo_map["district"].apply(normalize_geo)

# Applying corrected district names
recent_8_final["district"] = (
    recent_8_final["district"]
    .replace(district_mapping)
)

# Recreating matching keys
recent_8_final["state_key"] = (
    recent_8_final["state"].apply(normalize_geo)
)

recent_8_final["district_key"] = (
    recent_8_final["district"].apply(normalize_geo)
)

# Removing previous code columns
recent_8_final = recent_8_final.drop(
    columns=["state_code", "district_code"],
    errors="ignore"
)

# Merging LGD codes
recent_8_final = recent_8_final.merge(
    geo_map[
        [
            "state_key",
            "district_key",
            "lgd_statecode",
            "lgd_distcode"
        ]
    ],
    on=["state_key", "district_key"],
    how="left"
)

# Renaming codes
recent_8_final = recent_8_final.rename(
    columns={
        "lgd_statecode": "state_code",
        "lgd_distcode": "district_code"
    }
)

# Removing temporary columns
recent_8_final = recent_8_final[
    [
        "year",
        "state",
        "district",
        "state_code",
        "district_code",
        "crop",
        "season",
        "area_ha",
        "production_tonnes",
        "yield_kg_ha",
        "source"
    ]
]

print("Year values:")
print(sorted(recent_8_final["year"].unique()))

print("\nMissing state codes:",
      recent_8_final["state_code"].isna().sum())

print("Missing district codes:",
      recent_8_final["district_code"].isna().sum())

In [ ]:
#Building updated unified dataset

import pandas as pd

# Reloading original unified dataset
unified = pd.read_csv(
    r"D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv"
)

# Adding the corrected 8-crop UPAg data
unified_updated = pd.concat(
    [unified, recent_8_final],
    ignore_index=True
)

print("Original unified shape:", unified.shape)
print("Added recent 8-crop rows:", len(recent_8_final))
print("Updated unified shape:", unified_updated.shape)

print("\nYear values:")
print(sorted(unified_updated["year"].unique()))

In [ ]:
# Checking for duplicate records

key_cols = [
    "year",
    "state",
    "district",
    "crop",
    "season"
]

duplicates = unified_updated[
    unified_updated.duplicated(subset=key_cols, keep=False)
]

print("Duplicate records:", len(duplicates))

if len(duplicates) > 0:
    print("\nSample duplicates:")
    display(duplicates.head(20))
else:
    print("No duplicate records found.")

In [ ]:
#Crop-Year Coverage Check(12 crops x 12 years)

coverage = (
    unified_updated
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print("Crop-Year Coverage:")
display(coverage)

print("\nMissing crop-year combinations:")
missing = []

for crop in coverage.index:
    for year in coverage.columns:
        if coverage.loc[crop, year] == 0:
            missing.append((crop, year))

print("Total missing combinations:", len(missing))

if missing:
    for crop, year in missing:
        print(f"{crop} → {year}")
else:
    print("All crop-year combinations are present.")

In [ ]:
#Strict 12 × 12 Coverage Validation

expected_crops = [
    "Arhar/Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Maize",
    "Ragi",
    "Rice",
    "Soyabean",
    "Sugarcane",
    "Urad",
    "Wheat"
]

expected_years = [
    "2013-2014",
    "2014-2015",
    "2015-2016",
    "2016-2017",
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

actual_crops = sorted(unified_updated["crop"].unique())
actual_years = sorted(unified_updated["year"].unique())

print("Number of crops:", len(actual_crops))
print("Number of years:", len(actual_years))
print("Expected crop-year combinations:", 12 * 12)
print("Actual crop-year combinations:",
      unified_updated.groupby(["crop", "year"]).ngroups)

print("\nMissing crops:", set(expected_crops) - set(actual_crops))
print("Extra crops:", set(actual_crops) - set(expected_crops))

print("\nMissing years:", set(expected_years) - set(actual_years))
print("Extra years:", set(actual_years) - set(expected_years))

if (
    len(actual_crops) == 12
    and len(actual_years) == 12
    and set(actual_crops) == set(expected_crops)
    and set(actual_years) == set(expected_years)
    and unified_updated.groupby(["crop", "year"]).ngroups == 144
):
    print("\n✓ STRICT 12 CROPS × 12 YEARS CHECK PASSED")
else:
    print("\n✗ STRICT COVERAGE CHECK FAILED")

In [ ]:
#Final Dataset Summary

print("Dataset shape:", unified_updated.shape)

print("\nSource distribution:")
print(unified_updated["source"].value_counts())

print("\nCrop distribution:")
print(unified_updated["crop"].value_counts().sort_index())

print("\nData types:")
print(unified_updated.dtypes)

print("\nMissing values:")
print(unified_updated.isna().sum())

In [ ]:
#Saving Final Unified Dataset

output_path = (
    r"D:\Projects\AgriRisk and ROI Prediction\Dump"
    r"\data\processed\unified\unified_crop_yield_2013_2025.csv"
)

unified_updated.to_csv(output_path, index=False)

print("Final dataset saved successfully!")
print("Path:", output_path)
print("Shape:", unified_updated.shape)